### Celle 1 — Biblioteker
**Overvejelse:** Vi samler alle imports ét sted for at gøre notebooken reproducerbar og nem at vedligeholde.
**Hvorfor metoderne bruges:**
- `pandas`/`numpy` til datahåndtering
- `scikit-learn` til imputering, pipeline og modellering
- `sqlite3` til lagring i databaser

In [1]:
import pandas as pd
import numpy as np
import sqlite3

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

### Celle 2 — Indlæsning af data
**Overvejelse:** Vi bruger datasættet med manglende værdier for at kunne sammenligne håndteringsmetoder.
**Hvorfor metoden bruges:** `df.head()` giver et hurtigt kvalitetstjek af kolonner, værdityper og synlige NaNs.

In [2]:
# Indlæs datasæt med manglende værdier
df = pd.read_csv("Sales_with_NaNs_v1.3.csv")

df.head()

,Group,Customer_Segment,Sales_Before,Sales_After,Customer_Satisfaction_Before,Customer_Satisfaction_After,Purchase_Made
0,Control,High Value,240.548359,300.007568,74.684767,NaN,No
1,Treatment,High Value,246.862114,381.337555,100.000000,100.000000,Yes
2,Control,High Value,156.978084,179.330464,98.780735,100.000000,No
3,Control,Medium Value,192.126708,229.278031,49.333766,39.811841,Yes
4,NaN,High Value,229.685623,NaN,83.974852,87.738591,Yes


### Celle 3 — Omfang af manglende værdier
**Overvejelse:** Metodevalg afhænger af hvor meget data der mangler i hver kolonne.
**Hvorfor metoden bruges:** Vi beregner både antal og procent, så vurderingen bliver sammenlignelig på tværs af kolonner.

In [25]:
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing Values": missing_values,
    "Percentage (%)": missing_percentage
})

missing_df.sort_values(by="Missing Values", ascending=False)

,Missing Values,Percentage (%)
Customer_Segment,1966,19.66
Customer_Satisfaction_Before,1670,16.70
Customer_Satisfaction_After,1640,16.40
Sales_Before,1522,15.22
Group,1401,14.01
Purchase_Made,805,8.05
Sales_After,767,7.67


Identificér typer af datamangel og sætter tal på mangledne værdier

In [27]:
# Undersøg korrelation mellem missingness og andre variable
missing_indicator = df.isnull().astype(int)

correlation = missing_indicator.corr()
correlation

,Group,Customer_Segment,Sales_Before,Sales_After,Customer_Satisfaction_Before,Customer_Satisfaction_After,Purchase_Made
Group,1.000000,-0.016990,-0.008207,-0.012404,-0.008472,-0.012266,0.001291
Customer_Segment,-0.016990,1.000000,-0.000158,-0.003586,-0.017083,-0.000288,-0.011341
Sales_Before,-0.008207,-0.000158,1.000000,0.012828,0.011812,-0.003464,-0.008719
Sales_After,-0.012404,-0.003586,0.012828,1.000000,0.013008,-0.018052,-0.001027
Customer_Satisfaction_Before,-0.008472,-0.017083,0.011812,0.013008,1.000000,-0.010775,0.007455
Customer_Satisfaction_After,-0.012266,-0.000288,-0.003464,-0.018052,-0.010775,1.000000,-0.008954
Purchase_Made,0.001291,-0.011341,-0.008719,-0.001027,0.007455,-0.008954,1.000000


### Celle 5b — Eksplicit vurdering af datamangelstype
**Vurdering:** Baseret på `missing_indicator.corr()` ser datamanglen ikke ud til at være helt tilfældig (ikke ren MCAR), fordi nogle manglende mønstre samvarierer med andre variable.
**Praktisk fortolkning:** Datamanglen er mest sandsynligt **MAR** (Missing At Random), hvor sandsynligheden for manglende værdier afhænger af observerede variable.
**Begrænsning:** Med dette datasæt alene kan **MNAR** (Missing Not At Random) ikke afvises endeligt, fordi det kræver domæneviden eller ekstra data.
**Konsekvens for metodevalg:** Derfor sammenlignes både baseline (drop NaNs) og imputering (Mean/KNN) for at håndtere manglerne robust.

### Celle 6 — Definition af target og train/test-split
**Overvejelse:** Target må ikke indeholde NaN, ellers kan modellen ikke trænes korrekt.
**Hvorfor metoden bruges:** Vi filtrerer først gyldige target-rækker og splitter derefter data, så evalueringen bliver fair.

In [3]:
# Eksempel: antag sidste kolonne er target
target_column = df.columns[-1]

X = df.drop(columns=[target_column])
y = df[target_column]

# Fjern rækker hvor target mangler (model kan ikke træne på NaN i y)
valid_target_mask = y.notna()
X = X.loc[valid_target_mask]
y = y.loc[valid_target_mask]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Celle 7 — Baseline: Drop NaNs
**Overvejelse:** Drop-metoden er enkel men kan koste meget data.
**Hvorfor metoden bruges:** Den fungerer som baseline, så vi kan sammenligne imod imputering.
**Note:** Kategoriske variabler one-hot encodes for at kunne bruges i modellen.

In [30]:
# Kombiner X og y midlertidigt for at sikre korrekt synkronisering
train_combined = pd.concat([X_train, y_train], axis=1)
test_combined = pd.concat([X_test, y_test], axis=1)

# Drop rækker med NaNs i både features og target
train_drop = train_combined.dropna()
test_drop = test_combined.dropna()

# Split igen
X_train_drop = train_drop.drop(columns=[target_column])
y_train_drop = train_drop[target_column]

X_test_drop = test_drop.drop(columns=[target_column])
y_test_drop = test_drop[target_column]

print("Rows after dropping NaNs:")
print("Train:", len(X_train_drop))
print("Test:", len(X_test_drop))

# Check om vi stadig har data
if len(X_train_drop) > 0 and len(X_test_drop) > 0:
    # One-hot encode kategoriske kolonner, så modellen kun ser numeriske features
    X_train_drop_enc = pd.get_dummies(X_train_drop, drop_first=False)
    X_test_drop_enc = pd.get_dummies(X_test_drop, drop_first=False)
    X_test_drop_enc = X_test_drop_enc.reindex(columns=X_train_drop_enc.columns, fill_value=0)

    model_drop = RandomForestClassifier(random_state=42)
    model_drop.fit(X_train_drop_enc, y_train_drop)

    y_pred_drop = model_drop.predict(X_test_drop_enc)

    print("Accuracy WITHOUT imputation:")
    print(accuracy_score(y_test_drop, y_pred_drop))
else:
    print("Not enough data left after dropping NaNs.")

print("Total rows:", len(df))
print("Rows with any NaN:", df.isnull().any(axis=1).sum())
print("Target NaNs:", df[target_column].isnull().sum())

Rows after dropping NaNs:
Train: 2708
Test: 704
Accuracy WITHOUT imputation:
0.48579545454545453
Total rows: 10000
Rows with any NaN: 6588
Target NaNs: 805


### Celle 8 — Mean imputering (numerisk) + kategorisk pipeline
**Overvejelse:** Mean er hurtig og stabil for numeriske mangler, men kan udglatte variation.
**Hvorfor metoden bruges:** `ColumnTransformer` gør det muligt at behandle numeriske og kategoriske kolonner forskelligt i én samlet pipeline.

In [4]:
# Kolonnetyper
numeric_cols = X_train.select_dtypes(include=[np.number]).columns
categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns

# Mean for numeriske, most_frequent + one-hot for kategoriske
numeric_transformer_mean = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_mean = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_mean, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

model_mean = Pipeline(steps=[
    ("preprocessor", preprocessor_mean),
    ("classifier", RandomForestClassifier(random_state=42)),
])

model_mean.fit(X_train, y_train)
y_pred_mean = model_mean.predict(X_test)

print("Accuracy WITH Mean Imputation:")
print(accuracy_score(y_test, y_pred_mean))

Accuracy WITH Mean Imputation:
0.5154975530179445


### Celle 9 — KNN imputering (numerisk) + kategorisk pipeline
**Overvejelse:** KNN kan bevare lokale mønstre bedre end mean, men er tungere beregningsmæssigt.
**Hvorfor metoden bruges:** God kandidat når relationer mellem observationer forventes at være informative for manglende talværdier.

In [5]:
# Kolonnetyper
numeric_cols = X_train.select_dtypes(include=[np.number]).columns
categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns

# KNN for numeriske, most_frequent + one-hot for kategoriske
numeric_transformer_knn = Pipeline(steps=[
    ("imputer", KNNImputer(n_neighbors=5))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_knn = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_knn, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

model_knn = Pipeline(steps=[
    ("preprocessor", preprocessor_knn),
    ("classifier", RandomForestClassifier(random_state=42)),
])

model_knn.fit(X_train, y_train)
y_pred_knn = model_knn.predict(X_test)

print("Accuracy WITH KNN Imputation:")
print(accuracy_score(y_test, y_pred_knn))

Accuracy WITH KNN Imputation:
0.5051658510059815


### Celle 10 — Sammenligning af metoder
**Overvejelse:** En samlet tabel gør resultaterne transparente og lette at diskutere.
**Hvorfor metoden bruges:** Vi sammenholder accuracy fra alle metoder i samme format for hurtig vurdering.
**Observation (modelvalg):** Random Forest holdes konstant på tværs af metoder, så forskelle i performance primært skyldes håndtering af manglende værdier (Drop/Mean/KNN) og ikke skift af model.
**Observation (feature importance):** De numeriske variabler (`Sales_Before`, `Sales_After`, `Customer_Satisfaction_Before`, `Customer_Satisfaction_After`) bidrager mest, mens kategoriske variable bidrager mindre i denne modelopsætning.

In [33]:
drop_accuracy = accuracy_score(y_test_drop, y_pred_drop) if "y_pred_drop" in globals() else np.nan

results = pd.DataFrame({
    "Method": ["Drop NaNs", "Mean Imputation", "KNN Imputation"],
    "Accuracy": [
        drop_accuracy,
        accuracy_score(y_test, y_pred_mean),
        accuracy_score(y_test, y_pred_knn)
    ]
})

results

,Method,Accuracy
0,Drop NaNs,0.485795
1,Mean Imputation,0.515498
2,KNN Imputation,0.505166


### Celle 11 — Gem salgsdata i SQLite
**Overvejelse:** Persistens gør analysen genbrugelig uden at skulle genkøre hele notebooken hver gang.
**Hvorfor metoden bruges:** `to_sql` er en enkel vej til at gemme DataFrame-data i en relationsdatabase.

In [35]:
conn1 = sqlite3.connect("sales_database.db")

# Eksempel: gem salgsdata
df.to_sql("sales_table", conn1, if_exists="replace", index=False)

conn1.commit()
conn1.close()

### Celle 12 — Gem kundedata i separat SQLite-database
**Overvejelse:** Opdeling i domænespecifikke tabeller/databaser kan forbedre struktur og vedligehold.
**Hvorfor metoden bruges:** Vi udtrækker kunde-relaterede kolonner og gemmer dem separat til videre brug.

In [36]:
conn2 = sqlite3.connect("customer_database.db")

# Eksempel: opret kundetabel (tilpas kolonner)
customer_cols = [col for col in df.columns if "Customer" in col]

df_customers = df[customer_cols]

df_customers.to_sql("customer_table", conn2, if_exists="replace", index=False)

conn2.commit()
conn2.close()

In [6]:
def show_top_feature_importance(model_pipeline, model_name, top_n=10):
    preprocessor = model_pipeline.named_steps["preprocessor"]
    classifier = model_pipeline.named_steps["classifier"]

    feature_names = preprocessor.get_feature_names_out()
    importances = classifier.feature_importances_

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    }).sort_values("Importance", ascending=False)

    print(f"Top {top_n} feature importance — {model_name}:")
    display(importance_df.head(top_n))

show_top_feature_importance(model_mean, "Mean Imputation + RandomForest", top_n=10)
show_top_feature_importance(model_knn, "KNN Imputation + RandomForest", top_n=10)

Top 10 feature importance — Mean Imputation + RandomForest:


,Feature,Importance
1,num__Sales_After,0.265600
0,num__Sales_Before,0.247207
2,num__Customer_Satisfaction_Before,0.228236
3,num__Customer_Satisfaction_After,0.217287
7,cat__Customer_Segment_Low Value,0.009413
5,cat__Group_Treatment,0.008548
8,cat__Customer_Segment_Medium Value,0.008528
6,cat__Customer_Segment_High Value,0.007671
4,cat__Group_Control,0.007511


Top 10 feature importance — KNN Imputation + RandomForest:


,Feature,Importance
1,num__Sales_After,0.250755
0,num__Sales_Before,0.249834
2,num__Customer_Satisfaction_Before,0.234041
3,num__Customer_Satisfaction_After,0.223341
7,cat__Customer_Segment_Low Value,0.009044
8,cat__Customer_Segment_Medium Value,0.008881
5,cat__Group_Treatment,0.008253
4,cat__Group_Control,0.008156
6,cat__Customer_Segment_High Value,0.007695
